In [2]:
import pandas as pd
import numpy as np

In [3]:
noaa = pd.read_csv('../data/noaa.csv')
hhi = pd.read_csv('../data/hhi_dataset.csv')
disaster = pd.read_csv('../data/disaster.csv')

# Cleaning HHI dataset

In [4]:
hhi.head()

,STATEFP10,STATE,STATE_ABV,ZCTA,GEOID10,MULTI_STATE,POP,PR_HRI,F_HRI,LOW_EMS,...,P_RENT,PR_RENT,P_OZONE,PR_OZONE,P_PM25,PR_PM25,NBE_SCORE,NBE_RANK,OVERALL_SCORE,OVERALL_RANK
0,25,MA,Massachusetts,1001,2501001,0,16769,64,0.75,0,...,27.047079,0.6068,0.333333,0.4580,0.0,0.0,0.4645,0.5632,0.5703,0.6978
1,25,MA,Massachusetts,1002,2501002,0,29049,67,0.75,0,...,50.989998,0.9119,0.000000,0.0000,0.0,0.0,0.3750,0.3196,0.4838,0.5314
2,25,MA,Massachusetts,1003,2501003,0,10372,20,0.25,1,...,90.476190,0.9898,0.000000,0.0000,0.0,0.0,0.3494,0.2550,0.4246,0.4113
3,25,MA,Massachusetts,1005,2501005,0,5079,20,0.25,0,...,13.888889,0.2152,0.000000,0.0000,0.0,0.0,0.2338,0.0655,0.2391,0.0968
4,25,MA,Massachusetts,1007,2501007,0,14649,58,0.75,0,...,18.551141,0.3538,0.500000,0.6086,0.0,0.0,0.3866,0.3498,0.3194,0.2106


In [5]:
hhi.columns = hhi.columns.str.lower().str.strip()

In [6]:
# state fips -> "01", "08", ...
hhi["state_fips"] = hhi["statefp10"].astype(str).str.extract(r"(\d+)")[0].str.zfill(2)

# state abbrev -> "AL", "CO", ...
hhi["state"] = hhi["state"].astype(str).str.upper().str.strip()

# zcta -> "00501" style
hhi["zcta"] = hhi["zcta"].astype(str).str.extract(r"(\d+)")[0].str.zfill(5)

# geoid10 (often same as zcta) keep as string
hhi["geoid10"] = hhi["geoid10"].astype(str).str.extract(r"(\d+)")[0].str.zfill(5)

In [7]:
print(hhi["state"].head())
print(hhi["state_fips"].head())
print(hhi["zcta"].head())

0    MA
1    MA
2    MA
3    MA
4    MA
Name: state, dtype: object
0    25
1    25
2    25
3    25
4    25
Name: state_fips, dtype: object
0    01001
1    01002
2    01003
3    01005
4    01007
Name: zcta, dtype: object


In [8]:
non_numeric = {"state", "state_abv", "state_fips", "zcta", "geoid10", "multi_state"}
for c in hhi.columns:
    if c not in non_numeric:
        hhi[c] = pd.to_numeric(hhi[c], errors="coerce")

In [9]:
hhi = hhi.drop_duplicates(subset=["state_fips", "zcta"])
hhi = hhi.dropna(subset=["state", "state_fips", "zcta"])

In [10]:
keep = [
    "state","state_fips","zcta","pop",
    "p_pov","p_unemp","p_nohsdp","p_uninsur",
    "p_imperv","p_treec",
    "p_age65","p_asthma",
    "overall_score","hhb_score"
]
keep = [c for c in keep if c in hhi.columns]  # in case some names differ
hhi_clean = hhi[keep].copy()

In [11]:
feature_cols = [c for c in hhi_clean.columns if c not in ["state","state_fips","zcta","pop"]]

state_agg = (
    hhi_clean.dropna(subset=["pop"])
             .groupby(["state","state_fips"], as_index=False)
             .apply(lambda g: pd.Series({
                 **{c: np.average(g[c].dropna(), weights=g.loc[g[c].notna(), "pop"]) if g[c].notna().any() else np.nan
                    for c in feature_cols},
                 "pop_state_sum": g["pop"].sum()
             }))
)

state_agg.to_csv("hhi_clean_state.csv", index=False)

In [12]:
hhi.isna().sum()

statefp10        0
state            0
state_abv        0
zcta             0
geoid10          0
                ..
nbe_score        0
nbe_rank         0
overall_score    0
overall_rank     0
state_fips       0
Length: 76, dtype: int64

In [17]:
hhi_new = pd.read_csv('../data/hhi_clean.csv')
print(f"Rows: {hhi_new.shape[0]}, Columns: {hhi_new.shape[1]}")
hhi_new.head()


Rows: 49, Columns: 13


,state,state_fips,p_pov,p_unemp,p_nohsdp,p_uninsur,p_imperv,p_treec,p_age65,p_asthma,overall_score,hhb_score,pop_state_sum
0,AL,1,25.905201,5.914988,13.554414,9.515623,9.743966,46.650189,16.499473,10.640051,0.494570,0.394871,4844696.0
1,AR,5,28.227864,5.217266,12.941613,8.219707,7.857303,40.235269,16.908594,10.053575,0.552025,0.460649,2921346.0
2,AZ,4,24.761219,6.067404,13.288262,10.053251,30.664162,1.778124,17.309395,10.511697,0.663840,0.452221,6399084.0
3,CA,6,21.990544,6.119293,17.554979,7.388714,37.873581,7.534142,14.048054,9.591524,0.583420,0.174776,37269175.0
4,CO,8,16.111322,4.310988,8.377217,7.560296,24.659369,8.589071,13.852912,10.463270,0.553675,0.560786,5029092.0


# Cleaning Disaster Dataset

In [19]:
disaster.columns = disaster.columns.str.lower().str.strip()
disaster.head()

,femadeclarationstring,disasternumber,state,declarationtype,declarationdate,fydeclared,incidenttype,declarationtitle,ihprogramdeclared,iaprogramdeclared,...,placecode,designatedarea,declarationrequestnumber,lastiafilingdate,incidentid,region,designatedincidenttypes,lastrefresh,hash,id
0,FM-5529-OR,5529,OR,FM,2024-08-09T00:00:00.000Z,2024,Fire,LEE FALLS FIRE,0,0,...,99067,Washington (County),24122,NaN,2024081001,10,R,2024-08-27T18:22:14.800Z,ae87cf3c6ed795015b714af7166c7c295b2b67c7,09e3f81a-5e16-4b72-b317-1c64e0cfa59c
1,FM-5528-OR,5528,OR,FM,2024-08-06T00:00:00.000Z,2024,Fire,ELK LANE FIRE,0,0,...,99031,Jefferson (County),24116,NaN,2024080701,10,R,2024-08-27T18:22:14.800Z,432cf0995c47e3895cea696ede5621b810460501,59983f89-30bf-4888-b21b-62e8d57d9aac
2,FM-5527-OR,5527,OR,FM,2024-08-02T00:00:00.000Z,2024,Fire,MILE MARKER 132 FIRE,0,0,...,99017,Deschutes (County),24111,NaN,2024080301,10,R,2024-08-27T18:22:14.800Z,2f21d90cb6bc64b0d4121aa3f18d852bbb4b11fa,8d13ecf0-bc2f-496b-8c9f-b2e73da832a0
3,DR-4312-CA,4312,CA,DR,2017-05-02T00:00:00.000Z,2017,Severe Storm,FLOODING,0,0,...,60347,Resighini Rancheria (Indian Reservation),17035,NaN,2017041001,9,NaN,2025-03-26T20:21:32.579Z,432a3a64bdbb291ae26cf5a27a33deeabb380481,98a7c5bb-2346-45aa-a1ca-0399440d4f0b
4,DR-4251-AL,4251,AL,DR,2016-01-21T00:00:00.000Z,2016,Severe Storm,"SEVERE STORMS, TORNADOES, STRAIGHT-LINE WINDS,...",0,0,...,99001,Autauga (County),16003,NaN,2015122301,4,NaN,2025-03-27T12:21:46.559Z,dcd4ce6b37ee49875b3f1e32e9a8a16cd6a803d3,5229bbae-eee6-42b8-b277-edbafa8d6cb2


In [20]:
date_cols = ["declarationdate", "incidentbegindate", "incidentenddate"]
for c in date_cols:
    if c in disaster.columns:
        disaster[c] = pd.to_datetime(disaster[c], errors="coerce")

keep = [c for c in [
    "disasternumber", "state", "declarationdate",
    "incidenttype", "declarationtype", "declarationtitle",
    "fipsstatecode", "fipscountycode",
    "incidentbegindate", "incidentenddate"
] if c in disaster.columns]

disaster = disaster[keep].copy()

In [21]:
disaster["state"] = disaster["state"].astype(str).str.upper().str.strip()

if "fipsstatecode" in disaster.columns:
    disaster["state_fips"] = disaster["fipsstatecode"].astype("Int64").astype(str).str.zfill(2)
else:
    disaster["state_fips"] = pd.NA

if "fipscountycode" in disaster.columns:
    disaster["county_fips"] = disaster["fipscountycode"].astype("Int64").astype(str).str.zfill(3)
else:
    disaster["county_fips"] = pd.NA

# 5-digit county FIPS (only valid when county_fips is not missing/000)
disaster["full_fips"] = disaster["state_fips"].astype(str) + disaster["county_fips"].astype(str)

In [22]:
disaster = disaster.drop_duplicates()
disaster = disaster.dropna(subset=["state", "declarationdate", "incidenttype"])

In [23]:
import re
# heat-related flag (tune this list as you like)
heat_like = ["heat", "drought", "fire", "wildfire", "severe storm", "hurricane", "tropical storm"]
pat = "|".join([re.escape(x).replace("\\ ", "\\s+") for x in heat_like])
disaster["is_heat_related"] = disaster["incidenttype"].astype(str).str.lower().str.contains(pat, na=False).astype(int)

# county FIPS (optional)
disaster["state_fips"] = pd.to_numeric(disaster["fipsstatecode"], errors="coerce").astype("Int64").astype(str).str.zfill(2)
disaster["county_fips"] = pd.to_numeric(disaster["fipscountycode"], errors="coerce").astype("Int64").astype(str).str.zfill(3)
disaster["full_fips"] = disaster["state_fips"] + disaster["county_fips"]

# Aggregate
state_day = (
    disaster.groupby(["state", "declarationdate"], as_index=False)
      .agg(
          disasters_count=("disasternumber", "nunique"),  # unique disasters any type
          heat_related_disaster_count=("disasternumber", lambda s: s[disaster.loc[s.index, "is_heat_related"].eq(1)].nunique()),
          heat_related_area_count=("is_heat_related", "sum"),  # counts rows/areas
          heat_related_any=("is_heat_related", "max"),
          heat_related_counties_affected=("full_fips", lambda s: s[disaster.loc[s.index, "is_heat_related"].eq(1) & ~s.str.endswith("000", na=False)].nunique())
      )
      .rename(columns={"declarationdate": "date"})
)

In [24]:
print("Max heat_related_area_count:", state_day["heat_related_area_count"].max())
print("Max heat_related_disaster_count:", state_day["heat_related_disaster_count"].max())

Max heat_related_area_count: 255
Max heat_related_disaster_count: 10


In [25]:
print(f"Rows: {state_day.shape[0]}, Columns: {state_day.shape[1]}")
state_day.head()

Rows: 4772, Columns: 7


,state,date,disasters_count,heat_related_disaster_count,heat_related_area_count,heat_related_any,heat_related_counties_affected
0,AK,1953-10-30 00:00:00+00:00,1,0,0,0,0
1,AK,1954-11-10 00:00:00+00:00,1,0,0,0,0
2,AK,1955-12-22 00:00:00+00:00,1,0,0,0,0
3,AK,1964-03-28 00:00:00+00:00,1,0,0,0,0
4,AK,1967-08-17 00:00:00+00:00,1,0,0,0,0


In [38]:
print(state_day['heat_related_disaster_count'].mean())
print(state_day['heat_related_area_count'].mean())

0.7095557418273261
8.189019279128248


In [39]:
state_day.to_csv("disaster_clean.csv", index=False)

# Cleaning NOAA API Data

In [6]:
stations = pd.read_csv("../data/stations_sampled_10_per_state.csv")

stations_meta = stations[["id", "elevation"]].copy()

noaa = noaa.merge(stations_meta, how="left", left_on="station", right_on="id")

In [7]:
print("Missing elevation:", noaa["elevation"].isna().sum())

noaa.head()

Missing elevation: 0


,date,datatype,station,attributes,value,state,id_x,elevation_x,id_y,elevation_y,id,elevation
0,2020-01-01T00:00:00,TMAX,GHCND:USS0042N01S,",,T,",20.0,AK,GHCND:USS0042N01S,1011.9,GHCND:USS0042N01S,1011.9,GHCND:USS0042N01S,1011.9
1,2020-01-01T00:00:00,TMAX,GHCND:USS0049M08S,",,T,",18.0,AK,GHCND:USS0049M08S,716.3,GHCND:USS0049M08S,716.3,GHCND:USS0049M08S,716.3
2,2020-01-01T00:00:00,TMAX,GHCND:USS0049M22S,",,T,",17.0,AK,GHCND:USS0049M22S,634.0,GHCND:USS0049M22S,634.0,GHCND:USS0049M22S,634.0
3,2020-01-01T00:00:00,TMAX,GHCND:USW00013876,",,W,2400",56.0,AL,GHCND:USW00013876,187.7,GHCND:USW00013876,187.7,GHCND:USW00013876,187.7
4,2020-01-02T00:00:00,TMAX,GHCND:USS0042N01S,",,T,",13.0,AK,GHCND:USS0042N01S,1011.9,GHCND:USS0042N01S,1011.9,GHCND:USS0042N01S,1011.9


In [8]:
noaa = noaa.drop(columns=["id_x", "attributes", "id_y", "elevation_x", "elevation_y"])

In [9]:
noaa.head()

,date,datatype,station,value,state,id,elevation
0,2020-01-01T00:00:00,TMAX,GHCND:USS0042N01S,20.0,AK,GHCND:USS0042N01S,1011.9
1,2020-01-01T00:00:00,TMAX,GHCND:USS0049M08S,18.0,AK,GHCND:USS0049M08S,716.3
2,2020-01-01T00:00:00,TMAX,GHCND:USS0049M22S,17.0,AK,GHCND:USS0049M22S,634.0
3,2020-01-01T00:00:00,TMAX,GHCND:USW00013876,56.0,AL,GHCND:USW00013876,187.7
4,2020-01-02T00:00:00,TMAX,GHCND:USS0042N01S,13.0,AK,GHCND:USS0042N01S,1011.9


In [ ]:
noaa = noaa.sort_values(["state", "date", "station"])
noaa.head()

,date,datatype,station,value,state,id,elevation
0,2020-01-01T00:00:00,TMAX,GHCND:USS0042N01S,20.0,AK,GHCND:USS0042N01S,1011.9
1,2020-01-01T00:00:00,TMAX,GHCND:USS0049M08S,18.0,AK,GHCND:USS0049M08S,716.3
2,2020-01-01T00:00:00,TMAX,GHCND:USS0049M22S,17.0,AK,GHCND:USS0049M22S,634.0
4,2020-01-02T00:00:00,TMAX,GHCND:USS0042N01S,13.0,AK,GHCND:USS0042N01S,1011.9
5,2020-01-02T00:00:00,TMAX,GHCND:USS0049M08S,8.0,AK,GHCND:USS0049M08S,716.3


In [12]:
noaa = noaa.drop(columns=["datatype"])

noaa = noaa.rename(columns={"value": "TMAX"})

In [17]:
noaa.to_csv("noaa_clean.csv", index=False)

In [13]:
noaa.head()

,date,station,TMAX,state,id,elevation
0,2020-01-01T00:00:00,GHCND:USS0042N01S,20.0,AK,GHCND:USS0042N01S,1011.9
1,2020-01-01T00:00:00,GHCND:USS0049M08S,18.0,AK,GHCND:USS0049M08S,716.3
2,2020-01-01T00:00:00,GHCND:USS0049M22S,17.0,AK,GHCND:USS0049M22S,634.0
4,2020-01-02T00:00:00,GHCND:USS0042N01S,13.0,AK,GHCND:USS0042N01S,1011.9
5,2020-01-02T00:00:00,GHCND:USS0049M08S,8.0,AK,GHCND:USS0049M08S,716.3


# Merge 3 datasets to 1

In [28]:
noaa = pd.read_csv("../data/noaa_clean.csv")                 
dis  = pd.read_csv("../data/disaster_clean.csv")      
hhi  = pd.read_csv("../data/hhi_clean.csv")          

In [29]:
# Standardize state
for df in (noaa, dis, hhi):
    df["state"] = df["state"].astype(str).str.upper().str.strip()

# Standardize dates
noaa["date"] = pd.to_datetime(noaa["date"], errors="coerce").dt.date
dis["date"]  = pd.to_datetime(dis["date"], errors="coerce", utc=True).dt.date

In [30]:
US48_PLUS_DC = {
 'AL','AZ','AR','CA','CO','CT','DE','DC','FL','GA','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA',
 'MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD',
 'TN','TX','UT','VT','VA','WA','WV','WI','WY'
}

noaa = noaa[noaa["state"].isin(US48_PLUS_DC)].copy()
dis  = dis[dis["state"].isin(US48_PLUS_DC)].copy()
hhi  = hhi[hhi["state"].isin(US48_PLUS_DC)].copy()

In [31]:
merged = noaa.merge(dis, on=["state","date"], how="left")
merged = merged.merge(hhi, on="state", how="left")

In [32]:
fema_cols = [c for c in dis.columns if c not in ["state","date"]]
merged[fema_cols] = merged[fema_cols].fillna(0)

In [33]:
merged.isna().sum()

date                              0
station                           0
TMAX                              0
state                             0
id                                0
elevation                         0
disasters_count                   0
heat_related_disaster_count       0
heat_related_area_count           0
heat_related_any                  0
heat_related_counties_affected    0
state_fips                        0
p_pov                             0
p_unemp                           0
p_nohsdp                          0
p_uninsur                         0
p_imperv                          0
p_treec                           0
p_age65                           0
p_asthma                          0
overall_score                     0
hhb_score                         0
pop_state_sum                     0
dtype: int64

In [34]:
merged.head()

,date,station,TMAX,state,id,elevation,disasters_count,heat_related_disaster_count,heat_related_area_count,heat_related_any,...,p_unemp,p_nohsdp,p_uninsur,p_imperv,p_treec,p_age65,p_asthma,overall_score,hhb_score,pop_state_sum
0,2020-01-01,GHCND:USW00013876,56.0,AL,GHCND:USW00013876,187.7,0.0,0.0,0.0,0.0,...,5.914988,13.554414,9.515623,9.743966,46.650189,16.499473,10.640051,0.49457,0.394871,4844696.0
1,2020-01-02,GHCND:USW00013876,70.0,AL,GHCND:USW00013876,187.7,0.0,0.0,0.0,0.0,...,5.914988,13.554414,9.515623,9.743966,46.650189,16.499473,10.640051,0.49457,0.394871,4844696.0
2,2020-01-03,GHCND:USW00013876,65.0,AL,GHCND:USW00013876,187.7,0.0,0.0,0.0,0.0,...,5.914988,13.554414,9.515623,9.743966,46.650189,16.499473,10.640051,0.49457,0.394871,4844696.0
3,2020-01-04,GHCND:USW00013876,57.0,AL,GHCND:USW00013876,187.7,0.0,0.0,0.0,0.0,...,5.914988,13.554414,9.515623,9.743966,46.650189,16.499473,10.640051,0.49457,0.394871,4844696.0
4,2020-01-05,GHCND:USW00013876,57.0,AL,GHCND:USW00013876,187.7,0.0,0.0,0.0,0.0,...,5.914988,13.554414,9.515623,9.743966,46.650189,16.499473,10.640051,0.49457,0.394871,4844696.0


In [35]:
merged.shape

(196943, 23)

In [36]:
merged = merged.sort_values(["state","date","station"]).reset_index(drop=True)
merged.to_csv("../data/final_dataset.csv", index=False)